# 03F. Signal Reproducibility Engine

Test whether research-approved signals from Notebook 3E reproduce across alternative samples, subperiods, and universes. This notebook is diagnostic/final research gating only; it does not create signals or modify scoring, WFV, decay, regime, health, composite, alpha, stress/freeze, portfolio, or ML logic.


## 1. Purpose and scope

Use existing candidate signals, health gates, and clean close prices to evaluate reproducibility across deterministic universe slices and adaptive time subperiods.


## 2. Imports/config


In [1]:
from __future__ import annotations

import sys
from pathlib import Path

import pandas as pd
from IPython.display import display

cwd = Path.cwd().resolve()
project_root = next((path for path in [cwd, *cwd.parents] if (path / 'src').exists()), None)
if project_root is None:
    raise RuntimeError('Could not locate project root from current working directory.')

if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from src.db import get_db_path, load_price_table, load_table
from src.run_config import make_run_id, make_run_timestamp
from src.signal_reproducibility import (
    build_reproducibility_final_gate,
    build_reproducibility_subperiods,
    build_reproducibility_universes,
    load_reproducibility_candidates,
    run_signal_reproducibility_tests,
    summarize_signal_reproducibility,
)
from src.signal_reproducibility_storage import (
    SIGNAL_REPRODUCIBILITY_TABLES,
    save_signal_reproducibility_outputs,
)
from src.signal_storage import load_candidate_signals

REPRODUCIBILITY_VERSION = 'phase2_signal_reproducibility_v1'
INCLUDE_WATCHLIST = False
INCLUDE_ORTHOGONAL_WATCHLIST = True
ORTHOGONAL_WATCHLIST_MIN_HEALTH_SCORE = 60
ORTHOGONAL_WATCHLIST_VERSION = 'phase2_orthogonal_signals_v2'

sqlite_db_path = get_db_path()
print(f'SQLite database: {sqlite_db_path}')
print(f'Reproducibility version: {REPRODUCIBILITY_VERSION}')
print(f'Include watchlist: {INCLUDE_WATCHLIST}')
print(f'Include orthogonal watchlist: {INCLUDE_ORTHOGONAL_WATCHLIST}')
print(f'Orthogonal watchlist min health score: {ORTHOGONAL_WATCHLIST_MIN_HEALTH_SCORE}')
print(f'Orthogonal watchlist version: {ORTHOGONAL_WATCHLIST_VERSION}')


SQLite database: /Users/AnyiXu_1/Desktop/multi-factor-equity-alpha-model/sql/project_underdog.db
Reproducibility version: phase2_signal_reproducibility_v1
Include watchlist: False
Include orthogonal watchlist: True
Orthogonal watchlist min health score: 60
Orthogonal watchlist version: phase2_orthogonal_signals_v2


## 3. Create reproducibility run_id / timestamp


In [2]:
run_id = make_run_id('phase2_nb03f_signal_reproducibility')
run_timestamp = make_run_timestamp()

print(f'run_id: {run_id}')
print(f'run_timestamp: {run_timestamp}')


run_id: phase2_nb03f_signal_reproducibility_20260510_220339
run_timestamp: 2026-05-10 22:03:39


## 4. Load approved research signals from 3E


In [3]:
reproducibility_candidates = load_reproducibility_candidates(
    db_path=sqlite_db_path,
    include_watchlist=INCLUDE_WATCHLIST,
    include_orthogonal_watchlist=INCLUDE_ORTHOGONAL_WATCHLIST,
    orthogonal_watchlist_min_health_score=ORTHOGONAL_WATCHLIST_MIN_HEALTH_SCORE,
    orthogonal_watchlist_version=ORTHOGONAL_WATCHLIST_VERSION,
)
health_table = reproducibility_candidates.copy()

candidate_count = len(reproducibility_candidates)
candidate_tier_counts = (
    reproducibility_candidates["repro_candidate_tier"]
    .value_counts(dropna=False)
    .rename_axis("repro_candidate_tier")
    .reset_index(name="signal_horizon_count")
)
orthogonal_watchlist_candidates = reproducibility_candidates.loc[
    reproducibility_candidates["repro_candidate_tier"].eq("ORTHOGONAL_WATCHLIST_TEST")
].copy()
range_expansion_candidates = reproducibility_candidates.loc[
    reproducibility_candidates["signal_name"].eq("range_expansion_failure_5")
].copy()

print(f'Reproducibility candidate count: {candidate_count}')
print('Candidate counts by repro_candidate_tier')
display(candidate_tier_counts)
print('Orthogonal watchlist candidates admitted')
display(orthogonal_watchlist_candidates)
print('range_expansion_failure_5 candidate rows')
display(range_expansion_candidates)
display(reproducibility_candidates)


Reproducibility candidate count: 1
Candidate counts by repro_candidate_tier


,repro_candidate_tier,signal_horizon_count
0,ORTHOGONAL_WATCHLIST_TEST,1


Orthogonal watchlist candidates admitted


,signal_name,horizon,signal_family,signal_direction,signal_strength,best_mean_ic,best_abs_mean_ic,best_ic_ir,scoring_status,decay_status,...,signal_health_score,signal_health_gate,health_notes,run_id,health_version,signal_source,orthogonal_version,signal_version,orthogonal_cluster,repro_candidate_tier
0,vol_of_vol_20,10,volatility_structure,POSITIVE_EDGE,WEAK,0.015976,0.015976,0.128788,WATCHLIST,STABLE,...,62.0,WATCHLIST_RESEARCH,Mixed evidence; monitor before promotion,phase2_nb03e_signal_health_20260510_220324,phase2_signal_health_v1,orthogonal_generated,phase2_orthogonal_signals_v2,phase2_orthogonal_signals_v2,volatility_structure,ORTHOGONAL_WATCHLIST_TEST


range_expansion_failure_5 candidate rows


,signal_name,horizon,signal_family,signal_direction,signal_strength,best_mean_ic,best_abs_mean_ic,best_ic_ir,scoring_status,decay_status,...,signal_health_score,signal_health_gate,health_notes,run_id,health_version,signal_source,orthogonal_version,signal_version,orthogonal_cluster,repro_candidate_tier


,signal_name,horizon,signal_family,signal_direction,signal_strength,best_mean_ic,best_abs_mean_ic,best_ic_ir,scoring_status,decay_status,...,signal_health_score,signal_health_gate,health_notes,run_id,health_version,signal_source,orthogonal_version,signal_version,orthogonal_cluster,repro_candidate_tier
0,vol_of_vol_20,10,volatility_structure,POSITIVE_EDGE,WEAK,0.015976,0.015976,0.128788,WATCHLIST,STABLE,...,62.0,WATCHLIST_RESEARCH,Mixed evidence; monitor before promotion,phase2_nb03e_signal_health_20260510_220324,phase2_signal_health_v1,orthogonal_generated,phase2_orthogonal_signals_v2,phase2_orthogonal_signals_v2,volatility_structure,ORTHOGONAL_WATCHLIST_TEST


## 5. Load candidate_signals_current and clean close prices


In [4]:
close_prices = load_price_table('clean_close_prices_current', db_path=sqlite_db_path)
needed_signals = reproducibility_candidates['signal_name'].dropna().unique().tolist()

candidate_signals_long = load_candidate_signals(current=True, db_path=sqlite_db_path)
if needed_signals:
    candidate_signals_long = candidate_signals_long.loc[
        candidate_signals_long['signal_name'].isin(needed_signals)
    ].copy()

input_shapes = pd.DataFrame(
    [
        {'input_name': 'close_prices', 'rows': len(close_prices), 'columns': len(close_prices.columns)},
        {'input_name': 'candidate_signals_long_filtered', 'rows': len(candidate_signals_long), 'columns': len(candidate_signals_long.columns)},
    ]
)
display(input_shapes)


,input_name,rows,columns
0,close_prices,2098,478
1,candidate_signals_long_filtered,1002844,6


## 6. Build universes and subperiods


In [5]:
reproducibility_universes = build_reproducibility_universes(close_prices)
reproducibility_subperiods = build_reproducibility_subperiods(close_prices)

universe_summary = pd.DataFrame(
    [{'universe_name': name, 'n_tickers': len(tickers)} for name, tickers in reproducibility_universes.items()]
)
subperiod_summary = pd.DataFrame(
    [
        {'subperiod_name': name, 'start_date': start_date, 'end_date': end_date}
        for name, (start_date, end_date) in reproducibility_subperiods.items()
    ]
)

display(universe_summary)
display(subperiod_summary)


,universe_name,n_tickers
0,full_universe,477
1,first_half_tickers,238
2,second_half_tickers,239
3,random_half_seed_42,238
4,random_half_seed_99,238


,subperiod_name,start_date,end_date
0,full_period,2018-01-02,2026-05-07
1,early_period,2018-01-02,2020-02-03
2,middle_period,2020-02-03,2024-04-03
3,recent_period,2024-04-03,2026-05-07


## 7. Run reproducibility tests


In [6]:
reproducibility_results = run_signal_reproducibility_tests(
    candidate_table=reproducibility_candidates,
    candidate_signals_long=candidate_signals_long,
    close_prices=close_prices,
    run_id=run_id,
    reproducibility_version=REPRODUCIBILITY_VERSION,
)

failed_test_examples = reproducibility_results.loc[
    reproducibility_results['pass_flag'].eq(0)
].sort_values(['effective_mean_ic', 'positive_effective_ic_rate']).head(20)

detailed_result_shape = reproducibility_results.shape
print(f'Detailed result shape: {detailed_result_shape}')
display(reproducibility_results.head())


Detailed result shape: (14, 14)


,signal_name,horizon,test_type,test_name,n_obs,mean_ic_raw,effective_mean_ic,effective_ic_ir,positive_effective_ic_rate,abs_effective_mean_ic,pass_flag,failure_reason,run_id,reproducibility_version
0,vol_of_vol_20,10,universe,full_universe,577728,0.012808,0.012808,0.102073,0.531389,0.012808,1,passed,phase2_nb03f_signal_reproducibility_20260510_2...,phase2_signal_reproducibility_v1
1,vol_of_vol_20,10,universe,first_half_tickers,284746,0.009697,0.009697,0.070122,0.522491,0.009697,1,passed,phase2_nb03f_signal_reproducibility_20260510_2...,phase2_signal_reproducibility_v1
2,vol_of_vol_20,10,universe,second_half_tickers,292982,0.015041,0.015041,0.107335,0.532872,0.015041,1,passed,phase2_nb03f_signal_reproducibility_20260510_2...,phase2_signal_reproducibility_v1
3,vol_of_vol_20,10,universe,random_half_seed_42,270652,0.017087,0.017087,0.125060,0.545230,0.017087,1,passed,phase2_nb03f_signal_reproducibility_20260510_2...,phase2_signal_reproducibility_v1
4,vol_of_vol_20,10,universe,random_half_seed_99,294841,0.016653,0.016653,0.118421,0.536332,0.016653,1,passed,phase2_nb03f_signal_reproducibility_20260510_2...,phase2_signal_reproducibility_v1


## 8. Build reproducibility summary


In [7]:
reproducibility_summary = summarize_signal_reproducibility(reproducibility_results)
reproducibility_status_counts = reproducibility_summary['reproducibility_status'].value_counts() if not reproducibility_summary.empty else pd.Series(dtype=int)

display(reproducibility_summary)


,signal_name,horizon,n_tests,n_passed,pass_rate,worst_effective_mean_ic,avg_effective_mean_ic,min_positive_effective_ic_rate,failed_tests,reproducibility_status
0,vol_of_vol_20,10,14,12,0.857143,0.00472,0.015966,0.472868,"middle_period, first_half_tickers__recent_period",CONDITIONAL_PASS


## 9. Build final gate


In [8]:
reproducibility_gate = build_reproducibility_final_gate(
    summary=reproducibility_summary,
    health_table=health_table,
)
final_research_gate_counts = reproducibility_gate['final_research_gate'].value_counts() if not reproducibility_gate.empty else pd.Series(dtype=int)
reproducibility_status_counts = reproducibility_gate['reproducibility_status'].value_counts() if not reproducibility_gate.empty else pd.Series(dtype=int)
approved_alpha_research_signals = reproducibility_gate.loc[
    reproducibility_gate['final_research_gate'].eq('APPROVED_FOR_ALPHA_RESEARCH')
].sort_values(['signal_health_score', 'pass_rate'], ascending=[False, False])
range_expansion_gate_rows = reproducibility_gate.loc[
    reproducibility_gate['signal_name'].eq('range_expansion_failure_5')
].copy()

print('Reproducibility status counts')
display(reproducibility_status_counts.rename('signal_horizon_count'))
print('Final research gate counts')
display(final_research_gate_counts.rename('signal_horizon_count'))
print('range_expansion_failure_5 reproducibility rows')
display(range_expansion_gate_rows)
display(reproducibility_gate)


Reproducibility status counts


reproducibility_status
CONDITIONAL_PASS    1
Name: signal_horizon_count, dtype: int64

Final research gate counts


final_research_gate
WATCHLIST_ALPHA_RESEARCH    1
Name: signal_horizon_count, dtype: int64

range_expansion_failure_5 reproducibility rows


,signal_name,horizon,signal_family,repro_candidate_tier,signal_source,orthogonal_version,orthogonal_cluster,signal_version,signal_health_score,signal_health_gate,n_tests,n_passed,pass_rate,avg_effective_mean_ic,worst_effective_mean_ic,reproducibility_status,final_research_gate


,signal_name,horizon,signal_family,repro_candidate_tier,signal_source,orthogonal_version,orthogonal_cluster,signal_version,signal_health_score,signal_health_gate,n_tests,n_passed,pass_rate,avg_effective_mean_ic,worst_effective_mean_ic,reproducibility_status,final_research_gate
0,vol_of_vol_20,10,volatility_structure,ORTHOGONAL_WATCHLIST_TEST,orthogonal_generated,phase2_orthogonal_signals_v2,volatility_structure,phase2_orthogonal_signals_v2,62.0,WATCHLIST_RESEARCH,14,12,0.857143,0.015966,0.00472,CONDITIONAL_PASS,WATCHLIST_ALPHA_RESEARCH


## 10. Save outputs to SQLite


In [9]:
saved_paths = save_signal_reproducibility_outputs(
    reproducibility_results=reproducibility_results,
    reproducibility_summary=reproducibility_summary,
    reproducibility_gate=reproducibility_gate,
    db_path=sqlite_db_path,
    run_id=run_id,
    reproducibility_version=REPRODUCIBILITY_VERSION,
)

sqlite_tables_written = pd.DataFrame(
    [
        {
            'artifact': artifact,
            'current_table': tables[0],
            'history_table': tables[1],
            'sqlite_path': str(saved_paths[artifact]),
        }
        for artifact, tables in SIGNAL_REPRODUCIBILITY_TABLES.items()
    ]
)

display(sqlite_tables_written)


,artifact,current_table,history_table,sqlite_path
0,results,signal_reproducibility_results_current,signal_reproducibility_results_history,/Users/AnyiXu_1/Desktop/multi-factor-equity-al...
1,summary,signal_reproducibility_summary_current,signal_reproducibility_summary_history,/Users/AnyiXu_1/Desktop/multi-factor-equity-al...
2,gate,signal_reproducibility_gate_current,signal_reproducibility_gate_history,/Users/AnyiXu_1/Desktop/multi-factor-equity-al...


## 11. Final display


In [10]:
print('Candidate count')
display(pd.DataFrame([{'candidate_count': candidate_count}]))

print('Candidate counts by repro_candidate_tier')
display(candidate_tier_counts)

print('Orthogonal watchlist candidates admitted')
display(orthogonal_watchlist_candidates)

print('Detailed result shape')
display(pd.DataFrame([{'rows': detailed_result_shape[0], 'columns': detailed_result_shape[1]}]))

print('Reproducibility status counts')
display(reproducibility_status_counts.rename('signal_horizon_count'))

print('Final research gate counts')
display(final_research_gate_counts.rename('signal_horizon_count'))

print('range_expansion_failure_5 rows')
display(range_expansion_gate_rows)

print('Approved alpha research signals')
display(approved_alpha_research_signals)

print('Failed test examples')
display(failed_test_examples)

print('SQLite tables written')
display(sqlite_tables_written)


Candidate count


,candidate_count
0,1


Candidate counts by repro_candidate_tier


,repro_candidate_tier,signal_horizon_count
0,ORTHOGONAL_WATCHLIST_TEST,1


Orthogonal watchlist candidates admitted


...

,signal_name,horizon,signal_family,repro_candidate_tier,signal_source,orthogonal_version,orthogonal_cluster,signal_version,signal_health_score,signal_health_gate,n_tests,n_passed,pass_rate,avg_effective_mean_ic,worst_effective_mean_ic,reproducibility_status,final_research_gate


Failed test examples


,signal_name,horizon,test_type,test_name,n_obs,mean_ic_raw,effective_mean_ic,effective_ic_ir,positive_effective_ic_rate,abs_effective_mean_ic,pass_flag,failure_reason,run_id,reproducibility_version
10,vol_of_vol_20,10,universe_recent_period,first_half_tickers__recent_period,73207,0.004720,0.004720,0.037424,0.472868,0.004720,0,weak effective IC; weak sign consistency,phase2_nb03f_signal_reproducibility_20260510_2...,phase2_signal_reproducibility_v1
7,vol_of_vol_20,10,subperiod,middle_period,301667,0.005409,0.005409,0.039986,0.495710,0.005409,0,weak effective IC; weak sign consistency,phase2_nb03f_signal_reproducibility_20260510_2...,phase2_signal_reproducibility_v1


SQLite tables written


,artifact,current_table,history_table,sqlite_path
0,results,signal_reproducibility_results_current,signal_reproducibility_results_history,/Users/AnyiXu_1/Desktop/multi-factor-equity-al...
1,summary,signal_reproducibility_summary_current,signal_reproducibility_summary_history,/Users/AnyiXu_1/Desktop/multi-factor-equity-al...
2,gate,signal_reproducibility_gate_current,signal_reproducibility_gate_history,/Users/AnyiXu_1/Desktop/multi-factor-equity-al...


In [11]:
from src.db import load_table

results = load_table("signal_reproducibility_results_current")
summary = load_table("signal_reproducibility_summary_current")
gate = load_table("signal_reproducibility_gate_current")

print("results:", results.shape)
print("summary:", summary.shape)
print("gate:", gate.shape)

display(summary["reproducibility_status"].value_counts())
display(gate["final_research_gate"].value_counts())

display(
    gate.sort_values("pass_rate", ascending=False)
    [[c for c in ["signal_name", "horizon", "signal_family", "repro_candidate_tier", "signal_source", "orthogonal_version", "orthogonal_cluster", "signal_health_score",
      "pass_rate", "avg_effective_mean_ic", "worst_effective_mean_ic",
      "reproducibility_status", "final_research_gate"] if c in gate.columns]]
)


results: (14, 14)
summary: (1, 12)
gate: (1, 19)


reproducibility_status
CONDITIONAL_PASS    1
Name: count, dtype: int64

final_research_gate
WATCHLIST_ALPHA_RESEARCH    1
Name: count, dtype: int64

,signal_name,horizon,signal_family,repro_candidate_tier,signal_source,orthogonal_version,orthogonal_cluster,signal_health_score,pass_rate,avg_effective_mean_ic,worst_effective_mean_ic,reproducibility_status,final_research_gate
0,vol_of_vol_20,10,volatility_structure,ORTHOGONAL_WATCHLIST_TEST,orthogonal_generated,phase2_orthogonal_signals_v2,volatility_structure,62.0,0.857143,0.015966,0.00472,CONDITIONAL_PASS,WATCHLIST_ALPHA_RESEARCH


In [12]:
from src.db import load_table

results = load_table("signal_reproducibility_results_current")
gate = load_table("signal_reproducibility_gate_current")

display(
    results[results["pass_flag"] == 0]
    [["signal_name", "horizon", "test_type", "test_name",
      "n_obs", "effective_mean_ic", "positive_effective_ic_rate",
      "failure_reason"]]
)

display(
    gate.sort_values("pass_rate")
    [[c for c in ["signal_name", "horizon", "signal_family", "repro_candidate_tier", "signal_source", "orthogonal_version", "signal_health_score",
      "pass_rate", "worst_effective_mean_ic", "reproducibility_status",
      "final_research_gate"] if c in gate.columns]]
)


,signal_name,horizon,test_type,test_name,n_obs,effective_mean_ic,positive_effective_ic_rate,failure_reason
7,vol_of_vol_20,10,subperiod,middle_period,301667,0.005409,0.495710,weak effective IC; weak sign consistency
10,vol_of_vol_20,10,universe_recent_period,first_half_tickers__recent_period,73207,0.004720,0.472868,weak effective IC; weak sign consistency


,signal_name,horizon,signal_family,repro_candidate_tier,signal_source,orthogonal_version,signal_health_score,pass_rate,worst_effective_mean_ic,reproducibility_status,final_research_gate
0,vol_of_vol_20,10,volatility_structure,ORTHOGONAL_WATCHLIST_TEST,orthogonal_generated,phase2_orthogonal_signals_v2,62.0,0.857143,0.00472,CONDITIONAL_PASS,WATCHLIST_ALPHA_RESEARCH


In [13]:
results.groupby(["signal_name", "horizon"])["failure_reason"].value_counts()

signal_name    horizon  failure_reason                          
vol_of_vol_20  10       passed                                      12
                        weak effective IC; weak sign consistency     2
Name: count, dtype: int64

In [14]:
results.groupby("signal_name")["failure_reason"].nunique()

signal_name
vol_of_vol_20    2
Name: failure_reason, dtype: int64